# Project - AI for Medical Diagnosis and Prediction | Week #3

In this notebook, we continue our analysis of the MIMIC-CXR dataset by training our first classifiers. The objective is to perform a benchmark analysis of several machine learning classifiers to detect the presence of pathologies in chest x-rays.

We will use a subset of the **MIMIC-CXR dataset** **[1][2]**. The MIMIC Chest X-ray (MIMIC-CXR) Database v2.0.0 is a large, publicly available dataset of chest radiographs in DICOM format, accompanied by free-text radiology reports. It contains 377,110 images from 227,835 radiographic studies conducted at the Beth Israel Deaconess Medical Center in Boston, MA. The dataset has been de-identified in compliance with the US Health Insurance Portability and Accountability Act of 1996 (HIPAA) Safe Harbor requirements. All protected health information (PHI) has been removed. More details: [https://mimic.mit.edu/docs/iv/modules/cxr/](https://mimic.mit.edu/docs/iv/modules/cxr/)

<div class="alert alert-block alert-info">
<b>Your tasks are the following:</b>  <br>
- Create two subsets of the dataset: train and test <i>(Task 1)</i> <br>
- Prepare a list of models and hyperparameters to investigate during optimization <i>(Task 2)</i> <br>
- Implement the search using the cross-validation strategy of your choice <i>(Task 2*)</i> <br>
- Add an evaluation of data augmentation strategies as part of your optimization process <i>(Task 3)</i> <br>
- Run the benchmark analysis, save the results in a csv file using various appropriate metrics <i>(Task 4)</i> <br>
- Save the best model on the validation set <i>(Task 5)</i> <br>
- Compute the performance of this best model on the test set <i>(Task 5*)</i> <br>
</div>

**[1]** Johnson, A., Pollard, T., Mark, R., Berkowitz, S., & Horng, S. (2024). MIMIC-CXR Database (version 2.1.0). PhysioNet. [https://doi.org/10.13026/4jqj-jw95](https://doi.org/10.13026/4jqj-jw95).

**[2]** Johnson, A.E.W., Pollard, T.J., Berkowitz, S.J. et al. MIMIC-CXR, a de-identified publicly available database of chest radiographs with free-text reports. Sci Data 6, 317 (2019). [https://doi.org/10.1038/s41597-019-0322-0](https://doi.org/10.1038/s41597-019-0322-0)

In [ ]:
%pip install pydicom pynrrd -q
%pip install -q SimpleITK numpy PyWavelets

In [ ]:
import pandas as pd
import numpy as np
import pydicom
import nrrd

import SimpleITK as sitk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn import model_selection
import warnings

warnings.filterwarnings('ignore')

In [ ]:
DATA_PATH = './MIMIC-CXR'

In [ ]:
!curl https://uni-bonn.sciebo.de/s/XbomHCb6yL6nYN4/download/radiomics.csv --output ./radiomics.csv
!curl https://uni-bonn.sciebo.de/s/e7fKPxDYcs83J67/download/labels.csv --output ./labels.csv

If you do not have the dataset anymore, please re-run the notebook from week 2. Here, we will use the csv file created during week 2: `radiomics.csv` which contains input features and target classes.

In [ ]:
df = pd.read_csv(f'radiomics.csv')
labels_df = pd.read_csv(f'labels.csv')

In [ ]:
df.head()

## Task 1 - Dataset split
* Get the list of **UNIQUE** patients ;
* Split the list into train and test patients ;
* Extract two datasets: `train_df` and `test_df`

In [ ]:
patient_list = ... # COMPLETE

In [ ]:
from sklearn.model_selection import train_test_split

patient_train, patient_test = ... # COMPLETE

In [ ]:
train_df = df.loc[df['subject_id'].isin(patient_train)]
test_df = df.loc[df['subject_id'].isin(patient_test)]

train_labels_df = labels_df.loc[labels_df['subject_id'].isin(patient_train)]
test_labels_df = labels_df.loc[labels_df['subject_id'].isin(patient_test)]

# Save labels for future use
train_labels_df.to_csv('train_labels.csv', index=False)
test_labels_df.to_csv('test_labels.csv', index=False)

<div class="alert alert-block alert-info">
Visualize the repartition of classes across train and test set. Is there any imbalance?
</div>

In [ ]:
train_df['pathology'].value_counts()

In [ ]:
test_df['pathology'].value_counts()

In [ ]:
X_train = train_df.drop(['subject_id', 'study_id', 'dicom_id', 'dicom_path', 'labels_encoded', 'pathology'], axis=1)
y_train = train_df['pathology']

X_test = test_df.drop(['subject_id', 'study_id', 'dicom_id', 'dicom_path', 'labels_encoded', 'pathology'], axis=1)
y_test = test_df['pathology']

## Task 2 - Models and hyperparameters

* Define the list of models and hyperparameters you want to benchmark ;
* Implement the cross-validation strategy of your choice ;
* Create the searcher and run the benchmark analysis.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
import joblib
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler

# Define models and hyperparameter grids
models = {
    'Logistic Regression': (LogisticRegression(max_iter=1000, random_state=42), {
        'classifier__C': [0.01, 1, 10],
        'classifier__solver': ['lbfgs', 'liblinear']
    }),
    'Random Forest': (RandomForestClassifier(random_state=42), {
        'classifier__n_estimators': [10, 30, 50],
        'classifier__max_depth': [None, 5, 10]
    }),
    'Nearest Neighbors': (KNeighborsClassifier(), {
        'classifier__n_neighbors': [5, 10, 20]
    }),
    #'MLP': (MLPClassifier(), {
    #    'classifier__hidden_layer_sizes':[100, 50, 15],
    #    'classifier__alpha':[0.01, 0.1, 0.5, 1]
    #})
}

# Use StratifiedKFold for cross-validation to maintain class proportions
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

results = []
results_smote = []

In [ ]:
# Run benchmark with hyperparameter tuning (without SMOTE)
for name, (model, param_dist) in models.items():
    # Create a pipeline with StandardScaler and the classifier.
    pipeline = ImbPipeline([
        ('scaler', StandardScaler()),
        ('classifier', model)
    ])

    search = RandomizedSearchCV(pipeline, param_distributions=param_dist, n_iter=10,
                                scoring='recall', cv=cv, random_state=42, n_jobs=-1, error_score='raise')

    search.fit(X_train, y_train)
    best_model = search.best_estimator_
    best_score = search.best_score_
    std_dev = search.cv_results_['std_test_score'][search.best_index_]

    # Save best model into pkl format
    joblib.dump(best_model, f'best_model_{name.replace(" ", "_").lower()}.pkl')

    results.append({
        'Model': name,
        'Mean score': best_score,
        'Std Dev': std_dev
    })

## Task 3 - Data augmentation

* Add a data augmentation step in the pipeline, for instance using SMOTE ;
* Run the benchmark analysis using the data augmentation step.

In [ ]:
# Run benchmark with hyperparameter tuning, scaling, and SMOTE
for name, (model, param_dist) in models.items():
    # Create a full pipeline with StandardScaler, SMOTE, and the classifier.
    # This is the correct order of operations.
    pipeline = ImbPipeline([
        ('scaler', StandardScaler()),
        ('smote', SMOTE(random_state=42)),
        ('classifier', model)
    ])

    search = RandomizedSearchCV(pipeline, param_distributions=param_dist, n_iter=10,
                                scoring='recall', cv=cv, random_state=42, n_jobs=-1, error_score='raise')

    search.fit(X_train, y_train)

    # search.best_estimator_ is the entire tuned pipeline
    best_pipeline = search.best_estimator_
    best_score = search.best_score_
    std_dev = search.cv_results_['std_test_score'][search.best_index_]

    # Save the best pipeline (scaler + smote + model)
    joblib.dump(best_pipeline, f'best_model_smote_{name.replace(" ", "_").lower()}.pkl')

    results.append({
        'Model': name + ' SMOTE',
        'Mean score': best_score,
        'Std Dev': std_dev
    })

## Task 4 - Model selection

* Analyse the results provided by the benchmark analysis and select the best model ;
* Visualize different scores and comment.

In [ ]:
# Create results table
df_results = pd.DataFrame(results).sort_values(by='Mean Accuracy', ascending=False)

print("Benchmark Results:")
print(df_results)

# Optionally: save to CSV
#df_results.to_csv('model_benchmark_results_smote.csv', index=False)

<div class="alert alert-block alert-info">
Which model is the best performing one? <br>
Is there any benefits of data augmentation? <br>
Observe the metrics and comment on the expected behavior of the model for clinical practice.
</div>

## Task 5 - Evaluation

* Load the selected model ;
* Make a full performance report (ROC curve, metrics, confusion matrices) of the selected model.

In [ ]:
# Load a specific model (e.g., Logistic Regression without SMOTE)
best_model = joblib.load('...pkl') # COMPLETE

# Use it for predictions
predictions = best_model.predict(X_test)

# Get classification report
print(classification_report(y_test, predictions))

In [ ]:
from sklearn.metrics import DetCurveDisplay, RocCurveDisplay

fig, [ax_roc, ax_det] = plt.subplots(1, 2, figsize=(11, 5))

# Complete with other metrics and visualization
RocCurveDisplay.from_estimator(..., X_test, y_test, ax=ax_roc, name='ROC')
DetCurveDisplay.from_estimator(..., X_test, y_test, ax=ax_det, name='DET')

ax_roc.set_title("Receiver Operating Characteristic (ROC) curves")
ax_det.set_title("Detection Error Tradeoff (DET) curves")

ax_roc.grid(linestyle="--")
ax_det.grid(linestyle="--")

plt.legend()

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

predictions = best_model.predict(...)
cm = confusion_matrix(y_test, predictions, labels=best_model.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=...,
                              display_labels=best_model.classes_)
disp.plot()
plt.show()